Projection maps showing the multi-model mean change per degree of global warming, with stippling to indicate the level of model agreement on the direction of change. Based on Figure 3 of [Fischer et al 2014](https://doi.org/10.1002/2014GL062018).

In [1]:
import numpy as np
import xarray as xr

## Read precipitation data

In [2]:
hist_pr_file = 'pr_tavg-u-hxy-u_mon_glb_gr1.5_UKCM2a-0-HH_historical_r1i1p1f1_185001-202212.nc'
scen7h_pr_file = 'pr_tavg-u-hxy-u_mon_glb_gr1.5_UKCM2a-0-HH_scen7-h_r1i1p1f1_202201-210012.nc'

In [3]:
def calc_annual_precip(hist_file, scen_file):
    """Read precipitation data and calculate annual timeseries"""
    
    ds = xr.open_mfdataset([hist_file, scen_file], combine="nested", concat_dim="time")
    ds = ds.drop_duplicates(dim="time")
    ds['pr'] = ds['pr'] * 86400
    ds['pr'] = ds['pr'] * ds.time.dt.days_in_month
    ds['pr'].attrs['units'] = 'mm'
    annual_pr = ds['pr'].groupby('time.year').sum(dim='time', keep_attrs=True)
    
    return annual_pr

In [4]:
annual_pr = calc_annual_precip(hist_pr_file, scen7h_pr_file)
annual_pr = annual_pr.compute()

In [5]:
annual_pr

<xarray.DataArray 'pr' (year: 251, lat: 120, lon: 240)> Size: 58MB
array([[[ 89.5595559 ,  89.96068913,  90.56156608, ...,  89.1081447 ,
          89.26875371,  89.30289416],
        [ 94.86429578,  94.6551393 ,  94.48411656, ...,  96.76138218,
          95.77470709,  95.18801117],
        [ 83.39711198,  82.41382641,  81.24563825, ...,  85.1126539 ,
          85.02295536,  84.11176207],
        ...,
        [325.04166636, 322.01782455, 320.63532567, ..., 324.89589544,
         326.12157281, 326.71630603],
        [300.86414389, 300.51689999, 299.50171506, ..., 305.4854524 ,
         305.26640716, 302.53216277],
        [254.54319516, 254.35468446, 254.50698408, ..., 259.3351422 ,
         257.31702347, 255.31899515]],

       [[ 84.82029231,  84.83724005,  84.88813071, ...,  85.7509696 ,
          85.23636402,  84.97240151],
        [ 95.5273907 ,  95.59268921,  95.31204092, ...,  95.71743823,
          95.3944031 ,  95.39266559],
        [ 83.06083614,  82.59837319,  82.15883056, ...,  85.08570712,
          84.7457156 ,  83.94109074],
...
        [457.45872521, 457.49924213, 457.4775207 , ..., 457.65065145,
         457.37612575, 457.50219142],
        [401.53881729, 402.78121662, 404.16255128, ..., 397.03329277,
         397.73348552, 399.61075813],
        [351.06504333, 348.93379956, 347.13726538, ..., 346.271101  ,
         348.3825714 , 349.19842094]],

       [[109.42761049, 109.43617041, 109.39912976, ..., 110.59021622,
         110.10952816, 109.82314411],
        [109.67960946, 109.89852563, 109.79529358, ..., 110.73514792,
         110.02335306, 109.67537901],
        [ 88.56308825,  88.1914584 ,  88.47703606, ...,  91.85118365,
          91.17263159,  89.90874314],
        ...,
        [412.50195628, 417.20139724, 421.08745569, ..., 388.45970774,
         396.66465026, 403.86526483],
        [397.83627063, 392.48551983, 388.95777053, ..., 402.95881653,
         401.42804182, 401.7599647 ],
        [381.71122348, 379.49566859, 380.44591928, ..., 388.03465515,
         386.84189308, 384.83442223]]], shape=(251, 120, 240))
Coordinates:
  * year     (year) int64 2kB 1850 1851 1852 1853 1854 ... 2097 2098 2099 2100
  * lat      (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * lon      (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
Attributes:
    units:          mm
    original_name:  mo: (stash: m01s05i216, lbproc: 128)
    cell_methods:   area: time: mean
    cell_measures:  area: areacella
    bounds:         time_bnds
    axis:           T

## Read temperature data

In [2]:
hist_tas_file = 'tas_tavg-h2m-hxy-u_mon_glb_gr1.5_UKCM2a-0-HH_historical_r1i1p1f1_185001-202212.nc'
scen7h_tas_file = 'tas_tavg-h2m-hxy-u_mon_glb_gr1.5_UKCM2a-0-HH_scen7-h_r1i1p1f1_202201-210012.nc'

In [6]:
def calc_annual_gmst(hist_file, scen_file):
    """Read surface temperature data and calculate annual global mean surface temperature timeseries"""
    
    ds = xr.open_mfdataset([hist_file, scen_file], combine="nested", concat_dim="time")
    ds = ds.drop_duplicates(dim="time")

    spatial_weights = np.cos(np.deg2rad(ds['lat']))
    gmst = ds['tas'].weighted(spatial_weights).mean(dim=('lat', 'lon'), keep_attrs=True)

    days_in_month = gmst['time'].dt.days_in_month
    temporal_weights = (
        days_in_month.groupby('time.year') / days_in_month.groupby('time.year').sum()
    )
    annual_gmst = (gmst * temporal_weights).groupby('time.year').sum(dim='time', keep_attrs=True)

    annual_gmst.name = 'gmst'

    return annual_gmst

In [7]:
annual_gmst = calc_annual_gmst(hist_tas_file, scen7h_tas_file)
annual_gmst = annual_gmst.compute()

In [8]:
annual_gmst

<xarray.DataArray 'gmst' (year: 251)> Size: 2kB
array([287.17118464, 287.17385544, 287.20218289, 287.23063507,
       287.12193128, 286.97063922, 287.00997884, 286.93862916,
       287.0426161 , 286.95294921, 287.02003582, 286.99568841,
       286.86440678, 286.74789851, 286.8221852 , 286.99138462,
       287.11415447, 286.99673569, 287.03936463, 287.184373  ,
       286.86745077, 286.99165996, 286.94390045, 286.86875707,
       286.97590809, 287.06220141, 286.91866297, 287.06187184,
       287.0816731 , 286.97650967, 286.98753379, 287.00652422,
       287.04888258, 287.15030105, 286.93545723, 286.85437416,
       286.74443937, 286.70534477, 286.76820989, 286.9716916 ,
       287.03535168, 286.94811839, 287.10202967, 286.86268767,
       287.02294716, 286.98892474, 286.83102222, 287.02063793,
       287.0548358 , 286.9501493 , 286.91574365, 286.89997455,
       287.09918799, 286.97310052, 286.85417152, 286.96722013,
       286.92743008, 287.00800948, 287.15014647, 287.18532378,
       287.17958137, 287.23871218, 287.08941336, 286.98613811,
       287.02448112, 286.9587835 , 287.1302464 , 287.35855464,
       287.02633939, 287.07872913, 287.16521526, 286.99359283,
       287.15140668, 287.17104029, 287.01288818, 287.19671347,
       287.07446011, 287.14651886, 287.23004941, 287.24189816,
...
       288.60003418, 288.70333074, 288.75888475, 288.60246229,
       288.68822938, 288.7307293 , 288.62170514, 288.68633247,
       288.86341168, 288.83445236, 288.7507907 , 288.8856627 ,
       288.83032182, 288.88984313, 289.05138182, 288.89432201,
       289.0495781 , 289.24834063, 289.22906794, 289.21963226,
       289.1809181 , 289.19715729, 289.21715285, 289.32462565,
       289.37489985, 289.33764493, 289.38619956, 289.41034673,
       289.53088341, 289.51609267, 289.50489352, 289.54007301,
       289.56354501, 289.7758343 , 289.85983262, 289.83647514,
       289.84850198, 289.89569927, 289.7159822 , 289.77991417,
       289.83282238, 289.95222069, 290.03437761, 289.96521354,
       290.02422518, 289.99894123, 290.06926418, 290.35525704,
       290.36926172, 290.1251381 , 290.30565234, 290.49292189,
       290.50500985, 290.404774  , 290.41352816, 290.4727917 ,
       290.54108691, 290.67751121, 290.46379246, 290.47209128,
       290.49002476, 290.61513134, 290.63802415, 290.73379828,
       290.80619097, 290.67164824, 290.85951721, 290.9828014 ,
       290.91768668, 290.92036166, 290.93090263, 290.74198787,
       290.96556071, 291.16998124, 291.10554297, 291.17079506,
       291.05723368, 291.10461394, 291.3403514 ])
Coordinates:
  * year     (year) int64 2kB 1850 1851 1852 1853 1854 ... 2097 2098 2099 2100
Attributes:
    units:          K
    original_name:  mo: (stash: m01s03i236, lbproc: 128)
    cell_methods:   area: time: mean
    history:        2026-08-11T16:13:31Z altered by CMOR: Treated scalar dime...
    cell_measures:  area: areacella
    bounds:         time_bnds
    axis:           T

## Linear regression - precipitation vs GMST

In [9]:
pr_vs_gmst = annual_pr.assign_coords(gmst=('year', annual_gmst.data)).swap_dims({'year': 'gmst'})

In [10]:
pr_vs_gmst

<xarray.DataArray 'pr' (gmst: 251, lat: 120, lon: 240)> Size: 58MB
array([[[ 89.5595559 ,  89.96068913,  90.56156608, ...,  89.1081447 ,
          89.26875371,  89.30289416],
        [ 94.86429578,  94.6551393 ,  94.48411656, ...,  96.76138218,
          95.77470709,  95.18801117],
        [ 83.39711198,  82.41382641,  81.24563825, ...,  85.1126539 ,
          85.02295536,  84.11176207],
        ...,
        [325.04166636, 322.01782455, 320.63532567, ..., 324.89589544,
         326.12157281, 326.71630603],
        [300.86414389, 300.51689999, 299.50171506, ..., 305.4854524 ,
         305.26640716, 302.53216277],
        [254.54319516, 254.35468446, 254.50698408, ..., 259.3351422 ,
         257.31702347, 255.31899515]],

       [[ 84.82029231,  84.83724005,  84.88813071, ...,  85.7509696 ,
          85.23636402,  84.97240151],
        [ 95.5273907 ,  95.59268921,  95.31204092, ...,  95.71743823,
          95.3944031 ,  95.39266559],
        [ 83.06083614,  82.59837319,  82.15883056, ...,  85.08570712,
          84.7457156 ,  83.94109074],
...
        [457.45872521, 457.49924213, 457.4775207 , ..., 457.65065145,
         457.37612575, 457.50219142],
        [401.53881729, 402.78121662, 404.16255128, ..., 397.03329277,
         397.73348552, 399.61075813],
        [351.06504333, 348.93379956, 347.13726538, ..., 346.271101  ,
         348.3825714 , 349.19842094]],

       [[109.42761049, 109.43617041, 109.39912976, ..., 110.59021622,
         110.10952816, 109.82314411],
        [109.67960946, 109.89852563, 109.79529358, ..., 110.73514792,
         110.02335306, 109.67537901],
        [ 88.56308825,  88.1914584 ,  88.47703606, ...,  91.85118365,
          91.17263159,  89.90874314],
        ...,
        [412.50195628, 417.20139724, 421.08745569, ..., 388.45970774,
         396.66465026, 403.86526483],
        [397.83627063, 392.48551983, 388.95777053, ..., 402.95881653,
         401.42804182, 401.7599647 ],
        [381.71122348, 379.49566859, 380.44591928, ..., 388.03465515,
         386.84189308, 384.83442223]]], shape=(251, 120, 240))
Coordinates:
  * gmst     (gmst) float64 2kB 287.2 287.2 287.2 287.2 ... 291.1 291.1 291.3
  * lat      (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * lon      (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
    year     (gmst) int64 2kB 1850 1851 1852 1853 1854 ... 2097 2098 2099 2100
Attributes:
    units:          mm
    original_name:  mo: (stash: m01s05i216, lbproc: 128)
    cell_methods:   area: time: mean
    cell_measures:  area: areacella
    bounds:         time_bnds
    axis:           T

In [11]:
fit = pr_vs_gmst.polyfit(dim='gmst', deg=1)

In [14]:
fit

<xarray.Dataset> Size: 464kB
Dimensions:               (degree: 2, lat: 120, lon: 240)
Coordinates:
  * degree                (degree) int64 16B 1 0
  * lat                   (lat) float64 960B -89.25 -87.75 ... 87.75 89.25
  * lon                   (lon) float64 2kB 0.0 1.5 3.0 ... 355.5 357.0 358.5
Data variables:
    polyfit_coefficients  (degree, lat, lon) float64 461kB 5.832 ... -1.303e+04

In [12]:
slope = fit.polyfit_coefficients.sel(degree=1)
intercept = fit.polyfit_coefficients.sel(degree=0)

In [13]:
slope

<xarray.DataArray 'polyfit_coefficients' (lat: 120, lon: 240)> Size: 230kB
array([[ 5.83202682,  5.82128982,  5.82050012, ...,  5.92296336,
         5.88946325,  5.85524464],
       [ 6.2824334 ,  6.27702395,  6.27237017, ...,  6.35473865,
         6.31828856,  6.29416177],
       [ 4.87770339,  4.84460832,  4.87889596, ...,  4.9290095 ,
         4.91042354,  4.90248448],
       ...,
       [52.62548444, 53.42351753, 54.1772196 , ..., 50.78922069,
        51.28419375, 51.84436453],
       [50.43051197, 50.62927817, 50.88125867, ..., 49.80067202,
        49.96614841, 50.22352917],
       [46.12091459, 46.07106125, 46.13252128, ..., 46.11977545,
        46.14795869, 46.14688339]], shape=(120, 240))
Coordinates:
  * lat      (lat) float64 960B -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * lon      (lon) float64 2kB 0.0 1.5 3.0 4.5 6.0 ... 354.0 355.5 357.0 358.5
    degree   int64 8B 1

TODO: Plot slope over Australia (and compare against a regular time slice projection plot)